In [179]:
import os
import numpy as np 
import time 

import torch 
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
import torch.utils.data as data 

import torchvision.transforms as transforms 
import torchvision.datasets as datasets 

import matplotlib as plt 
from PIL import Image 

In [180]:
ROOT = './data'
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize([28, 28]),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

train_data = datasets.CIFAR10(
    root=ROOT,
    train=True,
    download=True,
    transform=transform,
)

test_data = datasets.CIFAR10(
    root=ROOT,
    train=False,
    download=True,
    transform=transform
)

In [181]:
VALID_RATIO = 0.9 

n_train_examples = int(len(train_data) * VALID_RATIO)
n_valid_examples = len(train_data) - n_train_examples

train_data, valid_data = data.random_split(
    train_data,
    [n_train_examples, n_valid_examples]
)

In [182]:
print(len(train_data))
print(len(valid_data))
print(len(test_data))

print(test_data.classes)
print()

for i in range(10):
    image, label = train_data[i]
    print(type(image))
    print(image.shape)
    print(label)
    print()

45000
5000
10000
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
6

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
6

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
4

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
2

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
9

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
5

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
6

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
8

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
6

<class 'torch.Tensor'>
torch.Size([3, 28, 28])
5



In [183]:
BATCH_SIZE = 64

train_dataloader = data.DataLoader(
    train_data, 
    shuffle=True,
    batch_size=BATCH_SIZE
)
valid_dataloader = data.DataLoader(
    valid_data, 
    batch_size=BATCH_SIZE
)

In [184]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(64, 64, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)
        
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.relu3 = nn.ReLU()
        self.conv4 = nn.Conv2d(128, 128, 3, padding=1)
        self.relu4 = nn.ReLU()
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)
        
        self.conv5 = nn.Conv2d(128, 256, 3, padding=1)
        self.relu5 = nn.ReLU()
        self.conv6 = nn.Conv2d(256, 256, 3, padding=1)
        self.relu6 = nn.ReLU()
        self.conv7 = nn.Conv2d(256, 256, 3, padding=1)
        self.relu7 = nn.ReLU()
        self.maxpool3 = nn.MaxPool2d(kernel_size=2)
        
        self.conv8 = nn.Conv2d(256, 256, 3, padding=1)
        self.relu8 = nn.ReLU()
        self.maxpool4 = nn.MaxPool2d(kernel_size=2)
        
        self.flatten = nn.Flatten()
        
        self.fc1 = nn.Linear(256, 128)
        self.relu14 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu15 = nn.ReLU()
        self.fc3 = nn.Linear(64, num_classes)
        
        
    def forward(self, inputs):
        outputs = self.conv1(inputs)
        outputs = self.relu1(outputs)
        outputs = self.conv2(outputs)
        outputs = self.relu2(outputs)
        outputs = self.maxpool1(outputs)
        
        outputs = self.conv3(outputs)
        outputs = self.relu3(outputs)
        outputs = self.conv4(outputs)
        outputs = self.relu4(outputs)
        outputs = self.maxpool2(outputs)
        
        outputs = self.conv5(outputs)
        outputs = self.relu5(outputs)
        outputs = self.conv6(outputs)
        outputs = self.relu6(outputs)
        outputs = self.conv7(outputs)
        outputs = self.relu7(outputs)
        outputs = self.maxpool3(outputs)
        
        outputs = self.conv8(outputs)
        outputs = self.relu8(outputs)    
        outputs = self.maxpool4(outputs)
        
        outputs = self.flatten(outputs)
        
        outputs = self.fc1(outputs)
        outputs = self.relu14(outputs)
        outputs = self.fc2(outputs)
        outputs = self.relu15(outputs)
        outputs = self.fc3(outputs)
        return outputs
        
        

In [185]:
def train(model, optimizer, criterion, train_dataloader, device, epoch=0, log_interval=50):
    model.train()
    total_acc = total_count = 0
    losses = []
    start_time = time.time()
    
    for idx, (inputs, labels) in enumerate(train_dataloader):
        inputs = inputs.to(device)
        # print(inputs.shape)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        predictions = model(inputs)
        
        loss = criterion(predictions, labels)
        losses.append(loss.item())
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        total_acc += (predictions.argmax(1) == labels).sum().item()
        
        total_count += labels.size(0)
        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time () - start_time
            # print(
            # "| epoch {:3d} | {:5d}/{:5d} batches "
            # "| accuracy {:8.3f}".format(epoch , idx , len(train_dataloader),total_acc / total_count)
            # )
            total_acc , total_count = 0, 0
            start_time = time.time ()
    epoch_acc = total_acc / total_count
    epoch_loss = sum(losses) / len(losses)
    return epoch_acc, epoch_loss

In [186]:
def evaluate(model, criterion, train_dataloader, device):
    model.eval()
    total_acc = total_count = 0
    losses = []
    
    with torch.no_grad():
        for idx, (inputs, labels) in enumerate(valid_dataloader):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            predictions = model(inputs)
            
            loss = criterion(predictions, labels)
            losses.append(loss.item())
            
            total_acc += (predictions.argmax(1) == labels).sum().item()
            total_count += labels.size(0)
        
    epoch_acc = total_acc / total_count
    epoch_loss = sum(losses) / len(losses)
    return epoch_acc, epoch_loss

In [187]:
num_classes = len(train_data.dataset.classes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
custom_model = Net(num_classes)
custom_model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(custom_model.parameters())

num_epochs = 10
save_model = './model'
os.makedirs(save_model, exist_ok=True)

train_accs, train_losses = [], []
eval_accs, eval_losses = [], []

best_loss_eval = 100

for epoch in range(1, num_epochs+1):
    epoch_start_time = time.time()
    
    train_acc, train_loss = train(custom_model, optimizer, criterion, train_dataloader, device, epoch)
    train_accs.append(train_acc)
    train_losses.append(train_loss)
    
    eval_acc, eval_loss = evaluate(custom_model, criterion, valid_dataloader, device)
    eval_accs.append(eval_acc)
    eval_losses.append(eval_loss)
    
    if eval_loss < best_loss_eval:
        torch.save(custom_model.state_dict(), save_model + '/custom_model.pt')
        
        
    print ( "-" * 59)
    print(
    "| End of epoch {:3d} | Time : {:5.2f}s | Train Accuracy {:8.3f} | Train Loss {:8.3f} "
    "| Valid Accuracy {:8.3f} | Valid Loss {:8.3f}".format(
        epoch,
        time.time() - epoch_start_time,
        train_acc,
        train_loss,
        eval_acc,
        eval_loss
        )
    )
    print ( "-" * 59)
    # Load best model
    custom_model.load_state_dict(torch.load(save_model + '/custom_model.pt'))
    custom_model.eval()

-----------------------------------------------------------
| End of epoch   1 | Time : 133.47s | Train Accuracy    0.500 | Train Loss    1.755 | Valid Accuracy    0.429 | Valid Loss    1.555
-----------------------------------------------------------
-----------------------------------------------------------
| End of epoch   2 | Time : 139.35s | Train Accuracy    0.669 | Train Loss    1.251 | Valid Accuracy    0.595 | Valid Loss    1.144
-----------------------------------------------------------
-----------------------------------------------------------
| End of epoch   3 | Time : 445.05s | Train Accuracy    0.699 | Train Loss    0.974 | Valid Accuracy    0.670 | Valid Loss    0.944
-----------------------------------------------------------
-----------------------------------------------------------
| End of epoch   4 | Time : 133.29s | Train Accuracy    0.779 | Train Loss    0.801 | Valid Accuracy    0.717 | Valid Loss    0.818
----------------------------------------------------

In [188]:
test_dataloader = data.DataLoader(
    test_data, 
    batch_size=BATCH_SIZE
)

test_acc, test_loss = evaluate(custom_model, criterion, test_dataloader, device)

test_acc, test_loss

(0.7672, 0.8146753224390971)

In [189]:
pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [190]:
from torchinfo import summary

summary(
    custom_model,
    input_size=(1, 3, 28, 28)
)

Layer (type:depth-idx)                   Output Shape              Param #
Net                                      [1, 10]                   --
├─Conv2d: 1-1                            [1, 64, 28, 28]           1,792
├─ReLU: 1-2                              [1, 64, 28, 28]           --
├─Conv2d: 1-3                            [1, 64, 28, 28]           36,928
├─ReLU: 1-4                              [1, 64, 28, 28]           --
├─MaxPool2d: 1-5                         [1, 64, 14, 14]           --
├─Conv2d: 1-6                            [1, 128, 14, 14]          73,856
├─ReLU: 1-7                              [1, 128, 14, 14]          --
├─Conv2d: 1-8                            [1, 128, 14, 14]          147,584
├─ReLU: 1-9                              [1, 128, 14, 14]          --
├─MaxPool2d: 1-10                        [1, 128, 7, 7]            --
├─Conv2d: 1-11                           [1, 256, 7, 7]            295,168
├─ReLU: 1-12                             [1, 256, 7, 7]         